<p align="center"><a href="https://www.plus2net.com/python/pandas-str-slice.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Pandas str.slice(): Extract and Validate Substrings

Run cells in order in Colab. All datasets are synthetic and embedded. Save a copy in Drive to keep edits. Try the exercise before its solution. Output formatting can vary by Pandas version.

## Start slicing at a character position

Positions begin at zero. <code>start=1</code> skips the first character. The source Series is unchanged.

In [ ]:
import pandas as pd
df = pd.DataFrame({"NAME": ["Ravi", "Raju", "Alex"]})
print(df["NAME"].str.slice(start=1))
assert df["NAME"].str.slice(start=1).tolist() == ["avi", "aju", "lex"]

**Expected output**

```text
0    avi
1    aju
2    lex
Name: NAME, dtype: str
```

## Count positions from the end

A negative start counts from the right. <code>start=-1</code> returns the last character, while <code>start=-2</code> returns up to two trailing characters.

In [ ]:
print(df["NAME"].str.slice(start=-1))
print(df["NAME"].str.slice(start=-2))
assert df["NAME"].str.slice(start=-1).tolist() == ["i", "u", "x"]

**Expected output**

```text
0    i
1    u
2    x
Name: NAME, dtype: str
0    vi
1    ju
2    ex
Name: NAME, dtype: str
```

## Stop is exclusive

<code>stop=2</code> selects positions zero and one. The character at position two is excluded. Omitting start uses the beginning for a positive step.

In [ ]:
print(df["NAME"].str.slice(stop=2))
assert df["NAME"].str.slice(stop=2).tolist() == ["Ra", "Ra", "Al"]

**Expected output**

```text
0    Ra
1    Ra
2    Al
Name: NAME, dtype: str
```

## Combine start and stop

<code>start=1, stop=3</code> selects positions one and two. Bounds describe positions within each string, not DataFrame rows.

In [ ]:
print(df["NAME"].str.slice(start=1, stop=3))
assert df["NAME"].str.slice(start=1, stop=3).tolist() == ["av", "aj", "le"]

**Expected output**

```text
0    av
1    aj
2    le
Name: NAME, dtype: str
```

## Select every second character

A positive step specifies the interval between selected positions. <code>step=2</code> starts at zero by default.

In [ ]:
print(df["NAME"].str.slice(step=2))
assert df["NAME"].str.slice(step=2).tolist() == ["Rv", "Rj", "Ae"]

**Expected output**

```text
0    Rv
1    Rj
2    Ae
Name: NAME, dtype: str
```

## Use start, stop and step together

The stop remains exclusive even when a step is supplied. Strings shorter than the stop are sliced only as far as their length.

In [ ]:
names = pd.Series(["RaviKing", "RajuQueen", "AlexMinister"])
print(names.str.slice(start=1, stop=10, step=2))
assert names.str.slice(start=1, stop=10, step=2).tolist() == ["aiig", "auue", "lxiit"]

**Expected output**

```text
0     aiig
1     auue
2    lxiit
dtype: str
```

## Compare slice() with str bracket notation

<code>s.str[1:3]</code> is shorthand for character slicing. <code>s.iloc[1:3]</code> selects Series rows instead. Bracket indexing is not a function call; see <a href="https://www.plus2net.com/python/pandas-dataframe-iloc.php">iloc row selection</a>.

In [ ]:
s = pd.Series(["Ravi", "Raju", "Alex"], index=[10, 20, 30], dtype="string")
print(s.str[1:3])
print("Rows selected with iloc:")
print(s.iloc[1:3])
assert s.str[1:3].equals(s.str.slice(1, 3))

**Expected output**

```text
10    av
20    aj
30    le
dtype: string
Rows selected with iloc:
20    Raju
30    Alex
dtype: string
```

## Reverse text with a negative step

<code>step=-1</code> reverses the character sequence when bounds are omitted. A zero step is invalid. Reversing a displayed word is a demonstration, not a general cleaning rule.

In [ ]:
s = pd.Series(["ABC123", "pen", ""], dtype="string")
print(s.str.slice(step=-1))
assert s.str.slice(step=-1).tolist() == ["321CBA", "nep", ""]

**Expected output**

```text
0    321CBA
1       nep
2          
dtype: string
```

## Handle missing values, blanks and short strings

Slicing preserves missing text. Out-of-range bounds produce shorter or empty strings rather than confirming valid input. Use <a href="https://www.plus2net.com/python/pandas-str-len.php">str.len()</a> when the source must have a required length.

In [ ]:
s = pd.Series(["AB1234", "A", "", None], dtype="string")
result = s.str.slice(start=2, stop=6)
print(result)
assert result.iloc[:3].tolist() == ["1234", "", ""]
assert pd.isna(result.iloc[3])

**Expected output**

```text
0    1234
1        
2        
3    <NA>
dtype: string
```

## Choose a policy for numeric input

Mixed object input is not reliable text data. Converting with <code>astype("string")</code> preserves missing entries and turns numbers into text deliberately. It cannot restore leading zeros already lost during numeric import. See <a href="https://www.plus2net.com/python/pandas-dataframe-astype.php">type conversion</a>.

In [ ]:
s = pd.Series(["0012", 1234, None], dtype="object")
text = s.astype("string")
print(text.str.slice(stop=2))
assert text.str.slice(stop=2).iloc[:2].tolist() == ["00", "12"]

**Expected output**

```text
0      00
1      12
2    <NA>
dtype: string
```

## Practical workflow: validate a fixed-format product code

This synthetic schema requires two uppercase ASCII letters followed by four digits. Trim surrounding whitespace, validate the entire value and then extract fields. Slicing alone would accept partial or malformed codes. Keep rejected values for review.

In [ ]:
raw = pd.DataFrame({"code": [" AB0012 ", "CD0456", "A123", "EF12X4", None, "GH0070extra"]})
code = raw["code"].astype("string").str.strip()
valid = code.str.fullmatch(r"[A-Z]{2}[0-9]{4}", na=False)
accepted = raw.loc[valid].copy()
accepted["code"] = code.loc[valid]
accepted["category"] = accepted["code"].str.slice(stop=2)
accepted["item_id"] = accepted["code"].str.slice(start=2, stop=6)
print(accepted)
print("Review rows:")
print(raw.loc[~valid])
assert accepted["item_id"].tolist() == ["0012", "0456"]

**Expected output**

```text
     code category item_id
0  AB0012       AB    0012
1  CD0456       CD    0456
Review rows:
          code
2         A123
3       EF12X4
4          NaN
5  GH0070extra
```

## Check extracted fields before export

Keep identifiers as text so their leading zeros remain intact. The checks below verify the schema after extraction. For delimiter-separated fields, prefer <a href="https://www.plus2net.com/python/pandas-str-split.php">str.split()</a>. For matching text, see <a href="https://www.plus2net.com/python/pandas-str-contains.php">str.contains()</a>.

In [ ]:
assert accepted["category"].str.len().eq(2).all()
assert accepted["item_id"].str.len().eq(4).all()
assert (accepted["category"] + accepted["item_id"]).equals(accepted["code"])
print(accepted.to_csv(index=False).rstrip())
# Optional Colab file export:
# accepted.to_csv("validated_code_fields.csv", index=False)

**Expected output**

```text
code,category,item_id
AB0012,AB,0012
CD0456,CD,0456
```

## Slice string labels in an Index

String slicing also works on an Index. If the result will become a key or column label, check uniqueness because shortening labels may create collisions.

In [ ]:
labels = pd.Index(["AB0012", "AB0456", "CD0070"], name="product")
prefixes = labels.str.slice(stop=2)
print(prefixes)
print("Prefixes unique:", prefixes.is_unique)
assert not prefixes.is_unique

**Expected output**

```text
Index(['AB', 'AB', 'CD'], dtype='str', name='product')
Prefixes unique: False
```

## Choose the right extraction method

Use character slicing when positions are defined by the data specification. Use <a href="https://www.plus2net.com/python/pandas-str-split.php">split()</a> when delimiters define fields, and regex extraction when structure varies. For dates, use <a href="https://www.plus2net.com/python/pandas-dt-to_datetime.php">to_datetime()</a> rather than assuming a fixed displayed date format. String positions are not UTF-8 byte offsets; combining characters can make displayed-character boundaries differ from simple slicing. Preserve readable names and international text.

## Common questions

<strong>Why is the final character missing?</strong> stop is exclusive. <strong>Why did I get an empty string?</strong> Check source length, bounds and step direction. <strong>Does slicing edit my column?</strong> No; assign the result explicitly if desired. <strong>Can every row have different bounds?</strong> This method takes shared bounds; use a deliberate per-row approach for variable offsets. <strong>Is str.slice() the same as iloc?</strong> No; one slices text and the other selects rows by position.

## Exercise: validate and extract a suffix

Create codes ORD-0012, ORD-0456, bad and a missing value. Require ORD- followed by exactly four digits. Extract the four-digit identifier and retain invalid rows for review. Expect two accepted codes.

## Exercise solution

Check the full format before extracting a suffix. The last four characters alone would not prove a valid prefix.

In [ ]:
orders = pd.DataFrame({"code": pd.Series(["ORD-0012", "ORD-0456", "bad", None], dtype="string")})
valid_order = orders["code"].str.fullmatch(r"ORD-[0-9]{4}", na=False)
answer = orders.loc[valid_order].copy()
answer["id"] = answer["code"].str.slice(start=-4)
print(answer)
print("Review rows:", int((~valid_order).sum()))
assert answer["id"].tolist() == ["0012", "0456"]

**Expected output**

```text
       code    id
0  ORD-0012  0012
1  ORD-0456  0456
Review rows: 2
```

## Practice in Google Colab

<a class="btn btn-outline-primary" href="https://colab.research.google.com/github/plus2net/pandas/blob/main/pandas_str_slice_data_cleaning.ipynb" target="_blank" rel="noopener">Open in Google Colab</a> <a class="btn btn-outline-secondary" href="https://github.com/plus2net/pandas/blob/main/pandas_str_slice_data_cleaning.ipynb" target="_blank" rel="noopener">View on GitHub</a><br>Run the examples in order, change slice bounds and sample codes, then compare extracted values and review rows. Save your own copy to keep edits. All sample tables are included in the notebook.